## Question 1

#### Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2  # For image processing
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow import keras
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
import os
from tensorflow.keras.optimizers import Adam
from PIL import Image


#### Data Loading

In [ ]:
def load_and_augment_data(data_dir, img_size=(64, 64)):
    images = []
    for filename in os.listdir(data_dir):
        img = cv2.imread(os.path.join(data_dir, filename), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size)
        images.append(img)

    images = np.array(images) / 255.0  # Normalize to [0, 1]

    # Data augmentation
    augmented_images = []
    for img in images:
        augmented_images.append(img)
        # Rotation
        for angle in [10, -10]:
            M = cv2.getRotationMatrix2D((img_size[0] / 2, img_size[1] / 2), angle, 1)
            rotated_img = cv2.warpAffine(img, M, img_size)
            augmented_images.append(rotated_img)

        # Adding noise
        noise = np.random.normal(0, 0.1, img.shape)
        noisy_img = np.clip(img + noise, 0, 1)
        augmented_images.append(noisy_img)

    return np.array(augmented_images)

#### Variational Autoencoder (VAE)

In [ ]:
class VAE(tf.keras.Model):
    def __init__(self, original_dim, latent_dim):
        super(VAE, self).__init__()
        self.encoder = models.Sequential([
            layers.Flatten(),
            layers.Dense(128, activation='relu'),
            layers.Dense(latent_dim + latent_dim)  # Mean and log variance
        ])

        self.decoder = models.Sequential([
            layers.Dense(128, activation='relu'),
            layers.Dense(original_dim, activation='sigmoid'),
            layers.Reshape((64, 64))  # Assuming output shape is 64x64
        ])

    def encode(self, x):
        mean_logvar = self.encoder(x)
        mean, logvar = tf.split(mean_logvar, num_or_size_splits=2, axis=1)
        return mean, logvar

    def reparameterize(self, mean, logvar):
        eps = tf.random.normal(shape=tf.shape(mean))
        return mean + tf.exp(0.5 * logvar) * eps

    def decode(self, z):
        return self.decoder(z)

    def call(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        return self.decode(z)

    def compute_loss(self, x, reconstructed_x, mean, logvar):
        reconstruction_loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(x, reconstructed_x))
        kl_loss = -0.5 * tf.reduce_mean(1 + logvar - tf.square(mean) - tf.exp(logvar))
        return reconstruction_loss + kl_loss

In [ ]:
def train_vae(vae, data, epochs=50, batch_size=32):
    optimizer = tf.keras.optimizers.Adam()

    for epoch in range(epochs):
        for i in range(0, len(data), batch_size):
            batch_data = data[i:i + batch_size]
            with tf.GradientTape() as tape:
                reconstructed = vae(batch_data)
                mean, logvar = vae.encode(batch_data)
                loss = vae.compute_loss(batch_data, reconstructed, mean, logvar)
            gradients = tape.gradient(loss, vae.trainable_variables)
            optimizer.apply_gradients(zip(gradients, vae.trainable_variables))
        print(f'Epoch {epoch + 1}, Loss: {loss.numpy()}')


#### Generating signatures

In [ ]:
def generate_signatures_vae(vae, num_samples=10):
    z_samples = tf.random.normal(shape=(num_samples, latent_dim))
    generated_signatures = vae.decode(z_samples)
    return generated_signatures.numpy()


In [ ]:
class GAN:
    def __init__(self, latent_dim):
        self.latent_dim = latent_dim

        # Build the generator
        self.generator = models.Sequential([
            layers.Dense(128, activation='relu', input_shape=(latent_dim,)),
            layers.Dense(64, activation='relu'),
            layers.Dense(64 * 64, activation='sigmoid'),  # Output shape for 64x64 images
            layers.Reshape((64, 64))
        ])

        # Build the discriminator
        self.discriminator = models.Sequential([
            layers.Flatten(input_shape=(64, 64)),
            layers.Dense(64, activation='relu'),
            layers.Dense(1, activation='sigmoid')
        ])

        self.discriminator.compile(optimizer='adam', loss='binary_crossentropy')

    def train(self, real_images, epochs=10000, batch_size=128):
        for epoch in range(epochs):
            # Train discriminator
            idx = np.random.randint(0, real_images.shape[0], batch_size)
            real_imgs = real_images[idx]

            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            fake_imgs = self.generator.predict(noise)

            d_loss_real = self.discriminator.train_on_batch(real_imgs, np.ones((batch_size, 1)))
            d_loss_fake = self.discriminator.train_on_batch(fake_imgs, np.zeros((batch_size, 1)))
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train generator
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            g_loss = self.discriminator.train_on_batch(self.generator.predict(noise), np.ones((batch_size, 1)))

            if epoch % 1000 == 0:
                print(f'Epoch {epoch}, D Loss: {d_loss}, G Loss: {g_loss}')


#### Through GAN

In [ ]:
def generate_signatures_gan(gan, num_samples=10):
    noise = np.random.normal(0, 1, (num_samples, gan.latent_dim))
    generated_signatures = gan.generator.predict(noise)
    return generated_signatures


In [ ]:
data_dir = '/content/dataset'
data = load_and_augment_data(data_dir)

original_dim = 64 * 64
latent_dim = 10

vae = VAE(original_dim, latent_dim)
train_vae(vae, data)
generated_signatures_vae = generate_signatures_vae(vae)

gan = GAN(latent_dim)
gan.train(data)

generated_signatures_gan = generate_signatures_gan(gan)


#### Visualize Generated Signatures

In [ ]:
def plot_generated_signatures(signatures, num_cols=5):
    num_images = signatures.shape[0]
    num_rows = num_images // num_cols + (num_images % num_cols > 0)

    plt.figure(figsize=(num_cols * 2, num_rows * 2))
    for i in range(num_images):
        plt.subplot(num_rows, num_cols, i + 1)
        plt.imshow(signatures[i], cmap='gray')
        plt.axis('off')
    plt.show()
plot_generated_signatures(generated_signatures_vae)
plot_generated_signatures(generated_signatures_gan)


## Question 2

#### Loading Dataset

In [ ]:
(x_train, y_train), (_, _) = cifar10.load_data()

cat_dog_indices = np.where((y_train == 3) | (y_train == 5))
x_train_filtered = x_train[cat_dog_indices]
y_train_filtered = y_train[cat_dog_indices]
x_train_filtered = x_train_filtered.astype('float32') / 255.0


#### Build Generator

In [ ]:
def build_generator(latent_dim):
    model = models.Sequential([
        layers.Dense(256, activation='relu', input_dim=latent_dim),
        layers.Dense(512, activation='relu'),
        layers.Dense(1024, activation='relu'),
        layers.Dense(32 * 32 * 3, activation='sigmoid'),  # CIFAR-10 images are 32x32x3
        layers.Reshape((32, 32, 3))
    ])
    return model

latent_dim = 100
generator = build_generator(latent_dim)


#### Train the GAN

In [ ]:
def train_gan(generator, discriminator, x_train, epochs=10000, batch_size=64):
    for epoch in range(epochs):
        idx = np.random.randint(0, x_train.shape[0], batch_size)
        real_images = x_train[idx]

        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        generated_images = generator.predict(noise)

        similarity_scores_real = discriminator.predict([real_images, real_images])
        similarity_scores_fake = discriminator.predict([real_images, generated_images])

        labels_real = np.ones((batch_size, 1))
        labels_fake = np.zeros((batch_size, 1))

        d_loss_real = discriminator.train_on_batch([real_images, real_images], labels_real)
        d_loss_fake = discriminator.train_on_batch([real_images, generated_images], labels_fake)

        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        labels_gan = np.ones((batch_size, 1))  # Generator wants the discriminator to think these are real
        g_loss = discriminator.train_on_batch([real_images, generator.predict(noise)], labels_gan)

        if epoch % 1000 == 0:
            print(f'Epoch {epoch}, D Loss Real: {d_loss_real}, D Loss Fake: {d_loss_fake}, G Loss: {g_loss}')

# Start training
train_gan(generator, discriminator, x_train_filtered)


## Question 3

#### Loading Data

In [ ]:
def load_images_from_folder(folder, target_size=(64, 64)):
    images = []
    for filename in os.listdir(folder):
        img = load_img(os.path.join(folder, filename), target_size=target_size, color_mode='grayscale')
        img = img_to_array(img)
        images.append(img)
    return np.array(images)

sketches = load_images_from_folder('path/to/sketches')
faces = load_images_from_folder('path/to/faces')

sketches = (sketches.astype(np.float32) - 127.5) / 127.5  # Scale to [-1, 1]
faces = (faces.astype(np.float32) - 127.5) / 127.5  # Scale to [-1, 1]
x_train, x_test, y_train, y_test = train_test_split(sketches, faces, test_size=0.2, random_state=42)

#### Define the cGAN Architecture

In [ ]:
def build_generator(latent_dim):
    model = models.Sequential()
    model.add(layers.Dense(128 * 16 * 16, activation="relu", input_dim=latent_dim))
    model.add(layers.Reshape((16, 16, 128)))
    model.add(layers.UpSampling2D())
    model.add(layers.Conv2D(128, kernel_size=3, padding='same', activation='relu'))
    model.add(layers.UpSampling2D())
    model.add(layers.Conv2D(64, kernel_size=3, padding='same', activation='relu'))
    model.add(layers.Conv2D(1, kernel_size=3, padding='same', activation='tanh'))  # Output shape (64, 64, 1)
    return model

def build_discriminator():
    input_sketch = layers.Input(shape=(64, 64, 1))
    input_face = layers.Input(shape=(64, 64, 1))
    merged = layers.Concatenate()([input_sketch, input_face])

    model = models.Sequential()
    model.add(layers.Conv2D(64, kernel_size=3, strides=2, padding='same', input_shape=(64, 64, 2)))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Conv2D(128, kernel_size=3, strides=2, padding='same'))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))

    return models.Model([input_sketch, input_face], model(merged))

#### Compile the cGAN Model

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
latent_dim = 100
generator = build_generator(latent_dim)
discriminator = build_discriminator()

discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
discriminator.trainable = False
gan_input = layers.Input(shape=(latent_dim,))
generated_face = generator(gan_input)
gan_output = discriminator([sketches, generated_face])
gan = models.Model(gan_input, gan_output)
gan.compile(optimizer='adam', loss='binary_crossentropy')


#### Train the cGAN

In [ ]:
def train_gan(epochs, batch_size):
    for epoch in range(epochs):
        idx = np.random.randint(0, x_train.shape[0], batch_size)
        real_faces = y_train[idx]
        sketches_batch = x_train[idx]
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        generated_faces = generator.predict(noise)
        d_labels_real = np.ones((batch_size, 1))
        d_labels_fake = np.zeros((batch_size, 1))
        d_loss_real = discriminator.train_on_batch([sketches_batch, real_faces], d_labels_real)
        d_loss_fake = discriminator.train_on_batch([sketches_batch, generated_faces], d_labels_fake)
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_labels = np.ones((batch_size, 1))  # Generator wants the discriminator to think these are real

        g_loss = gan.train_on_batch(noise, g_labels)
        if epoch % 100 == 0:
            print(f'Epoch {epoch} | D Loss: {0.5 * np.add(d_loss_real[0], d_loss_fake[0]):.4f}, G Loss: {g_loss:.4f}')
train_gan(epochs=5000, batch_size=32)


#### Images

In [ ]:
def generate_image_from_sketch(sketch):
    sketch = (sketch.astype(np.float32) - 127.5) / 127.5  # Normalize to [-1, 1]
    noise = np.random.normal(0, 1, (1, latent_dim))
    generated_face = generator.predict(noise)
    return (generated_face[0] * 127.5 + 127.5).astype(np.uint8)  # Scale back to [0, 255]

# Example usage
generated_face_image = generate_image_from_sketch(sketches[0])  # Replace with your sketch
plt.imshow(generated_face_image, cmap='gray')
plt.axis('off')
plt.show()


## Question 4

#### Loading Dataset

In [ ]:
def load_dataset(image_dir, sketch_dir):
    # Load images and sketches
    images = []
    sketches = []

    for filename in os.listdir(image_dir):
        img_path = os.path.join(image_dir, filename)
        sketch_path = os.path.join(sketch_dir, filename)

        if os.path.isfile(img_path) and os.path.isfile(sketch_path):
            image = Image.open(img_path).resize((256, 256))
            sketch = Image.open(sketch_path).resize((256, 256))
            images.append(np.array(image))
            sketches.append(np.array(sketch))

    return np.array(images), np.array(sketches)

# Example usage
image_dir = '/content/images'
sketch_dir = '/content/sketch'
faces, sketches = load_dataset(image_dir, sketch_dir)


#### Define the CycleGAN Model Architecture

In [ ]:
def build_generator():
    model = models.Sequential()
    model.add(layers.Input(shape=(256, 256, 3)))
    model.add(layers.Conv2D(64, (7, 7), padding='same'))
    model.add(layers.ReLU())
    return model

def build_discriminator():
    model = models.Sequential()
    model.add(layers.Input(shape=(256, 256, 3)))
    model.add(layers.Conv2D(64, (3, 3), strides=2, padding='same'))
    model.add(layers.LeakyReLU(alpha=0.2))
    return model
generator_g = build_generator()  # Face to Sketch
generator_f = build_generator()  # Sketch to Face
discriminator_x = build_discriminator()  # Discriminator for Faces
discriminator_y = build_discriminator()  # Discriminator for Sketches


#### Define the CycleGAN Training Loop

In [ ]:
def compile_models(generator_g, generator_f, discriminator_x, discriminator_y):
    # Compile models with optimizers
    generator_g.compile(optimizer=Adam(2e-4, beta_1=0.5))
    generator_f.compile(optimizer=Adam(2e-4, beta_1=0.5))
    discriminator_x.compile(optimizer=Adam(2e-4, beta_1=0.5))
    discriminator_y.compile(optimizer=Adam(2e-4, beta_1=0.5))

def train_cyclegan(epochs, batch_size):
    for epoch in range(epochs):
        for i in range(0, len(faces), batch_size):
            real_faces = faces[i:i + batch_size]
            real_sketches = sketches[i:i + batch_size]

            fake_sketches = generator_g.predict(real_faces)
            fake_faces = generator_f.predict(real_sketches)

        if (epoch + 1) % 10 == 0:
            generator_g.save_weights('generator_g_weights_epoch_{}.h5'.format(epoch + 1))
            generator_f.save_weights('generator_f_weights_epoch_{}.h5'.format(epoch + 1))

compile_models(generator_g, generator_f, discriminator_x, discriminator_y)
train_cyclegan(epochs=100, batch_size=16)

#### User Interface with Flask

In [ ]:
from flask import Flask, request, jsonify
from flask import render_template

app = Flask(__name__)

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    # Load the image from the request
    file = request.files['image']
    image = Image.open(file.stream).resize((256, 256))
    input_image = np.array(image) / 255.0

    # Generate face from sketch or sketch from face based on input
    if 'is_sketch' in request.form:  # Example flag to determine the type
        output_image = generator_f.predict(np.expand_dims(input_image, axis=0))  # Sketch to Face
    else:
        output_image = generator_g.predict(np.expand_dims(input_image, axis=0))  # Face to Sketch

    output_image = (output_image[0] * 255).astype(np.uint8)
    output_image_pil = Image.fromarray(output_image)
    output_image_path = 'output_image.png'
    output_image_pil.save(output_image_path)

    return jsonify({'output_image_path': output_image_path})

if __name__ == '__main__':
    app.run(debug=True)
